In [ ]:
import pandas as pd
df=pd.read_csv("Medicine_Details__Enhanced__Data.csv")
df.head()

,Medicine Name,Composition,Uses,Side_effects,Image URL,Manufacturer,Excellent Review %,Average Review %,Poor Review %
0,Avastin 400mg Injection,Bevacizumab (400mg),Cancer of colon and rectum Non-small cell lun...,Rectal bleeding Taste change Headache Noseblee...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Roche Products India Pvt Ltd,22,56,22
1,Augmentin 625 Duo Tablet,Amoxycillin (500mg) + Clavulanic Acid (125mg),Treatment of Bacterial infections,Vomiting Nausea Diarrhea Mucocutaneous candidi...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glaxo SmithKline Pharmaceuticals Ltd,47,35,18
2,Azithral 500 Tablet,Azithromycin (500mg),Treatment of Bacterial infections,Nausea Abdominal pain Diarrhea,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Alembic Pharmaceuticals Ltd,39,40,21
3,Ascoril LS Syrup,Ambroxol (30mg/5ml) + Levosalbutamol (1mg/5ml)...,Treatment of Cough with mucus,Nausea Vomiting Diarrhea Upset stomach Stomach...,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Glenmark Pharmaceuticals Ltd,24,41,35
4,Aciloc 150 Tablet,Ranitidine (150mg),Treatment of Gastroesophageal reflux disease (...,Headache Diarrhea Gastrointestinal disturbance,"https://onemg.gumlet.io/l_watermark_346,w_480,...",Cadila Pharmaceuticals Ltd,34,37,29


In [6]:
print("Number of rows",df.shape[0])
print("Number of columns:",df.shape[1])
print("/ncolumn names:")
print(df.columns.tolist())

Number of rows 11825
Number of columns: 9
/ncolumn names:
['Medicine Name', 'Composition', 'Uses', 'Side_effects', 'Image URL', 'Manufacturer', 'Excellent Review %', 'Average Review %', 'Poor Review %']


In [4]:
df.isnull().sum()

Medicine Name         0
Composition           0
Uses                  0
Side_effects          0
Image URL             0
Manufacturer          0
Excellent Review %    0
Average Review %      0
Poor Review %         0
dtype: int64

In [7]:
# Combine important medicine information into one searchable field

df["search_text"] = (
    df["Medicine Name"].astype(str) + " " +
    df["Composition"].astype(str) + " " +
    df["Uses"].astype(str) + " " +
    df["Side_effects"].astype(str) + " " +
    df["Manufacturer"].astype(str)
)

# Convert to lowercase
df["search_text"] = df["search_text"].str.lower()

print("Searchable text created successfully!")
df[["Medicine Name", "search_text"]].head()

Searchable text created successfully!


,Medicine Name,search_text
0,Avastin 400mg Injection,avastin 400mg injection bevacizumab (400mg) c...
1,Augmentin 625 Duo Tablet,augmentin 625 duo tablet amoxycillin (500mg) ...
2,Azithral 500 Tablet,azithral 500 tablet azithromycin (500mg) treat...
3,Ascoril LS Syrup,ascoril ls syrup ambroxol (30mg/5ml) + levosal...
4,Aciloc 150 Tablet,aciloc 150 tablet ranitidine (150mg) treatment...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(df["search_text"])

print("TF-IDF completed!")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF completed!
Matrix shape: (11825, 42188)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search_medicine(query, top_n=5):

    # Convert query to TF-IDF
    query_vector = vectorizer.transform([query.lower()])

    # Calculate similarity
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Get highest scores
    top_indices = np.argsort(similarity_scores)[::-1][:top_n]

    # Create results
    results = df.iloc[top_indices].copy()

    results["Similarity Score"] = similarity_scores[top_indices]

    return results

In [13]:
results = search_medicine(query, 5)

results[
    [
        "Medicine Name",
        "Composition",
        "Uses",
        "Side_effects",
        "Similarity Score"
    ]
]

,Medicine Name,Composition,Uses,Side_effects,Similarity Score
9114,Solvin Cough Tablet,Chlorpheniramine Maleate (2mg) + Dextromethorp...,Treatment of Dry cough,Upset stomach Sleepiness,0.234651
11422,Zedex Cough Syrup,Chlorpheniramine Maleate (2mg) + Dextromethorp...,Treatment of Dry cough,Upset stomach Sleepiness,0.232682
9167,Solvin Cough Syrup Sugar Free,Chlorpheniramine Maleate (2mg) + Dextromethorp...,Treatment of Dry cough,Upset stomach Sleepiness,0.222408
7591,Protussa Cough Syrup,Chlorpheniramine Maleate (1mg/5ml) + Dextromet...,Treatment of Dry cough,Upset stomach Sleepiness,0.220023
466,Alex Cough Lozenges Sugar Free,Dextromethorphan Hydrobromide (5mg),Dry cough,Sleepiness Dizziness Confusion Nausea,0.215598


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Create searchable text
df["search_text"] = (
    df["Medicine Name"].astype(str) + " " +
    df["Composition"].astype(str) + " " +
    df["Uses"].astype(str) + " " +
    df["Side_effects"].astype(str) + " " +
    df["Manufacturer"].astype(str)
).str.lower()

# Create TF-IDF model
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(df["search_text"])

# Search function
def search_medicine(query, top_n=5):

    query_vector = vectorizer.transform([query.lower()])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(similarity_scores)[::-1][:top_n]

    results = df.iloc[top_indices].copy()

    results["Similarity Score"] = similarity_scores[top_indices]

    return results


# Interactive search
print("=" * 70)
print("       💊 MEDICINE INFORMATION RETRIEVAL SYSTEM")
print("=" * 70)

query = input("Enter your search query: ")

results = search_medicine(query, 5)

print("\nTop 5 Relevant Results:\n")

for i, (_, row) in enumerate(results.iterrows(), 1):

    print("-" * 70)
    print("Result", i)
    print("Medicine Name :", row["Medicine Name"])
    print("Composition   :", row["Composition"])
    print("Uses          :", row["Uses"])
    print("Side Effects  :", row["Side_effects"])
    print("Similarity    :", round(row["Similarity Score"], 4))

       💊 MEDICINE INFORMATION RETRIEVAL SYSTEM


Enter your search query:  leg pain



Top 5 Relevant Results:

----------------------------------------------------------------------
Result 1
Medicine Name : Pioglar-G Tablet
Composition   : Pioglitazone (15mg) + Glimepiride (2mg)
Uses          : Treatment of Type 2 diabetes mellitus
Side Effects  : Edema swelling Cold symptoms Headache Urinary tract infection Diarrhea Nausea Leg pain Decreased blood cells red cells white cells and platelets Infection Accidental injury
Similarity    : 0.2357
----------------------------------------------------------------------
Result 2
Medicine Name : Zotide Injection
Composition   : Teriparatide (750mcg)
Uses          :  Osteoporosis
Side Effects  : Joint pain Injection site reactions pain swelling redness Nausea Leg cramps
Similarity    : 0.1175
----------------------------------------------------------------------
Result 3
Medicine Name : Terifrac Injection
Composition   : Teriparatide (750mcg)
Uses          :  Osteoporosis
Side Effects  : Joint pain Injection site reactions pain swe

#### ABSTRACT
* Medicine Information Retrieval System is developed to retrieve relevant medicine information from a large medicine dataset based on user queries. The system uses medicine name, composition, uses, side effects, and manufacturer details for information retrieval. Text preprocessing is performed to prepare the data for searching. TF-IDF is used for text representation and Cosine Similarity is used to calculate the relevance between the query and medicine records. The results are ranked according to similarity scores and the most relevant records are displayed. The project is implemented using Python, Pandas, and Scikit-learn.